Original code from CompleteP: https://github.com/EleutherAI/nanoGPT-mup/blob/completep/completep_examples/relative_distance_from_linearization.ipynb


# Collect results

In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


In [2]:
import os
# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
# os.environ["CUDA_VISIBLE_DEVICES"]="5"

In [3]:
import torch
import torch.nn as nn
from typing import Optional

def get_layer(model, layer_of_interest_name):
    return dict(model.named_modules())[layer_of_interest_name]

def get_param(model, param_of_interest_name):
    return dict(model.named_parameters())[param_of_interest_name]

def get_activation_and_grad_wrt_param(
    model: nn.Module,
    inputs: torch.Tensor,
    layer_of_interest_name: str,
    param_of_interest: torch.Tensor,
    return_vector=False
):
    # We'll store the activation from the forward hook
    activation: Optional[torch.Tensor] = None
    
    def forward_hook(module, in_t, out_t):
        nonlocal activation
        activation = out_t

    # Find the module by name
    layer_of_interest = get_layer(model, layer_of_interest_name)

    # Register hook
    hook_handle = layer_of_interest.register_forward_hook(forward_hook)

    # Forward pass
    model.zero_grad(set_to_none=True)
    out = model(inputs)  # Could be logits or next layer's hidden states, doesn't matter
    hook_handle.remove()

    # Now `activation` should hold h^l
    if activation is None:
        raise RuntimeError(f"No activation captured from layer {layer_of_interest_name}.")

    # For the simplest approach: we do d( sum(h^l) ) / d param
    # That is effectively a shape match if we pass grad_outputs = ones_like(activation).
    grad_outputs = torch.ones_like(activation[0,0], requires_grad=True)

    # Get derivative of the [0,0] entry (first batch, first nueron).
    grad_tuple = torch.autograd.grad(
        outputs=activation[0,0],    # or out if we want ∂out/∂W, but here we do activation
        inputs=param_of_interest,
        grad_outputs=grad_outputs,
        retain_graph=False,  # if we want to do more grads afterwards
        create_graph=False,  # if we don't need higher order derivatives
        allow_unused=True
    )
    grad = grad_tuple[0]

    # Return both the captured h^l and the gradient
    return activation[0,0] if not return_vector else activation[0,:], grad


def get_first_order_taylor(delta_w, grad):
    return torch.sum(delta_w[0,:] * grad[0:])


In [4]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# A minimal test model
class SimpleLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, x):
        phi_x = torch.relu(x)  # This is your φ(x)
        return self.linear(phi_x)

def test_derivative_wrt_first_row():
    torch.manual_seed(0)
    in_dim = 4
    out_dim = 3

    device =  "cuda" if torch.cuda.is_available() else "cpu"
    model = SimpleLayer(in_dim=in_dim, out_dim=out_dim)
    model.to(device)

    # Input x
    x = torch.randn(1, in_dim, requires_grad=True, device=device)

    # Get param and grad
    W = get_param(model, "linear.weight")
    phi_x = torch.relu(x)
    act, grad = get_activation_and_grad_wrt_param(
        model=model,
        inputs=x,
        layer_of_interest_name="linear",
        param_of_interest=W
    )

    # The derivative of (W φ(x))_0 w.r.t. W is a matrix whose first row is φ(x), others are 0
    expected_grad = torch.zeros_like(W)
    expected_grad[0, :] = phi_x

    assert torch.allclose(grad, expected_grad, atol=1e-6), f"Gradient mismatch:\nGot: {grad}\nExpected: {expected_grad}"

    print("Test passed!")

# Run the test
test_derivative_wrt_first_row()

Test passed!


In [5]:
import math 

# Define a residual branch
class ResidualBranch(nn.Module):
    def __init__(self, N, K):
        super().__init__()
        self.layers = nn.ModuleList()
        self.activs = nn.ModuleList()
        self.N = N
        for _ in range(K):
            self.activs.append(nn.ReLU())
            self.layers.append(nn.Linear(N,N,bias=False))
        self.reset_parameters()

    def forward(self, x):
        for layer, activ in zip(self.layers, self.activs):
            x = activ(x)
            x = math.sqrt(2/self.N) * layer(x)
        return x
    

    def reset_parameters(self) -> None:
        for layer in self.layers:   
            with torch.no_grad():
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
                # layer.weight.normal_(0, 1)
                nn.init.orthogonal_(layer.weight, gain=1.0)
            
# Toy residual model   
class ToyModel(nn.Module):
    def __init__(self, D, N_out, N, L, K, alpha, base_depth):
        super(ToyModel, self).__init__()
        
        self.D = D
        self.L = L
        self.N = N
        self.alpha = alpha
        self.base_depth = base_depth
        
        self.activations = []
        
        self.readin = nn.Linear(D, N, bias=False)
        
        self.branches = nn.ModuleList()
        
        for _ in range(L):
            branch = ResidualBranch(N, K)
            self.branches.append(branch)
            
        self.readout = nn.Linear(N,N_out,bias=False)

        self.reset_parameters()
    
    def forward(self, x):
        self.activations = []
        x = 1/math.sqrt(self.D ) * self.readin(x)
        
        for branch in self.branches:
            x = x + 1/((self.L/self.base_depth)**self.alpha) * branch(x)
            self.activations.append(x)
        
        x = 1/(self.N) * self.readout(x)
        
        return x
    
    # def reset_parameters(self) -> None:
    #     for layer in [self.readin, self.readout]:
    #         with torch.no_grad():
    #             if layer.bias is not None:
    #                 nn.init.zeros_(layer.bias)
    #             layer.weight.normal_(0,1)
    def reset_parameters(self) -> None:
        for layer in [self.readin, self.readout]:
            with torch.no_grad():
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
                nn.init.orthogonal_(layer.weight, gain=1.0)
        
        for branch in self.branches:
            branch.reset_parameters()

In [ ]:
from spectral_clip import spectral_clip_jordan, spectral_clip_jiacheng, spectral_clip_jianlin, mclip3_jiacheng

# -----------------------------------------------------------------------------
# Scion optimizer

def zeroth_power_via_svd(G):
   U, S, V = G.svd()
   return U @ V.T

@torch.compile
def zeropower_via_newtonschulz5(G, steps=5):
    """
    Newton-Schulz iteration to compute the zeroth power / orthogonalization of G. We opt to use a
    quintic iteration whose coefficients are selected to maximize the slope at zero. For the purpose
    of minimizing steps, it turns out to be empirically effective to keep increasing the slope at
    zero even beyond the point where the iteration no longer converges all the way to one everywhere
    on the interval. This iteration therefore does not produce UV^T but rather something like US'V^T
    where S' is diagonal with S_{ii}' ~ Uniform(0.5, 1.5), which turns out not to hurt model
    performance at all relative to UV^T, where USV^T = G is the SVD.
    """
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750,  2.0315)
    X = G.bfloat16()
    if G.size(0) > G.size(1):
        X = X.T
    # Ensure spectral norm is at most 1
    X = X / (X.norm() + 1e-7)
    # Perform the NS iterations
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A # adapted from suggestion by @jxbz, @leloykun, and @YouJiacheng
        X = a * X + B @ X
    
    if G.size(0) > G.size(1):
        X = X.T
    return X

class Norm(object):
    def lmo(self, g):
        raise NotImplementedError

class Spectral(Norm):
    def __init__(self, steps=5):
        self.steps = steps
    def lmo(self, g):
        g = zeropower_via_newtonschulz5(g.reshape(len(g), -1), steps=self.steps).view(g.shape)
        d_out, d_in = g.shape
        g *= (d_out / d_in)**0.5
        return g

class Sign(Norm):
    def __init__(self, zero_init=False):
        self.zero_init = zero_init
    def lmo(self, g):
        out_channels, in_channels = g.shape     # in_channels=768
        return (1/in_channels)*torch.sign(g)    

norm_dict = {
    'Spectral': Spectral,
    'Sign': Sign
}

class Scion(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, momentum=1.0, norm: str='Spectral', norm_kwargs: dict=None, scale=1.0, unconstrained=False, projected=False, spectral_clipping_type="Jiacheng"):
        if lr < 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if momentum < 0.0:
            raise ValueError(f"Invalid momentum value: {momentum}")
        defaults = dict(lr=lr, momentum=momentum, scale=scale, unconstrained=unconstrained)
        super().__init__(params, defaults)
        self.projected = projected
        self.spectral_clipping_type = spectral_clipping_type

    def step(self):
        for group in self.param_groups:
            lr = group['lr']
            momentum = group['momentum']
            scale = group['scale']
            unconstrained = group['unconstrained']
            norm_backend = norm_dict[group['norm']](**group['norm_kwargs'])
            projected = self.projected
            spectral_clipping_type = self.spectral_clipping_type
            for p in group['params']:
                g = p.grad
                if g is None:
                    continue
                state = self.state[p]
                if momentum != 1:
                    if 'momentum_buffer' not in state.keys():
                        state['momentum_buffer'] = torch.zeros_like(g)
                    buf = state['momentum_buffer']
                    buf.mul_(1-momentum).add_(g, alpha=momentum)
                    g = buf
                
                update = scale * norm_backend.lmo(g)

                # add the projection option
                if not unconstrained:
                    if projected: # spectral clipping projection.
                        if spectral_clipping_type=='Jiacheng':
                            p.data.add_(update, alpha=-lr)
                            p.data = spectral_clip_jiacheng(p.data, beta=scale)
                        elif spectral_clipping_type=='Jordan':
                            p.data.add_(update, alpha=-lr)
                            p.data = spectral_clip_jordan(p.data, beta=scale)
                    else: # constrained with convex combination
                        p.data.mul_(1-lr)
                        p.data.add_(update, alpha=-lr)
                else: # unconstrained
                    p.data.add_(update, alpha=-lr)
                    # print("update done")

In [7]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np

seed = 101
torch.manual_seed(seed)
np.random.seed(seed)

transform = transforms.Compose([
    transforms.ToTensor(),  # Convert PIL to Tensor
    transforms.Normalize((0.1307,), (0.3081,)),  # Normalize to mean and std of MNIST
    transforms.Lambda(lambda x: x.view(-1))  # Flatten to shape: [784]
])

train_dataset = datasets.MNIST(root='/home/ep42de8l/data/', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root='/home/ep42de8l/data/', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [10]:
from tqdm import tqdm
import numpy as np


def find_best_lr(alpha, scion_method, lrs, n_steps=10):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    losses = {}
    L = 4

    for lr in tqdm(lrs):
        model = ToyModel(D, N_out, N, L, K, alpha, base_depth).to(device)

        linear_weights = []
        first_weight = []
        last_weight = []

        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                if name == "readout":
                    last_weight.append(module.weight)
                elif name == "readin":
                    first_weight.append(module.weight)
                else:
                    linear_weights.append(module.weight)

        # print("linear weight:", linear_weights)
        optim_groups = [
            {
                'params': linear_weights,
                'norm': 'Spectral',
                'norm_kwargs': {'steps': 9},
                'scale': 1.0
            },
            {
                'params': first_weight,
                'norm': 'Spectral',
                'norm_kwargs': {},
                'scale': 1.0
            },
            {
                'params': last_weight,
                'norm': 'Sign',
                'norm_kwargs': {},
                'scale': 1.0
            }
        ]

        param_0 = get_param(model, weight_of_interest_name)
        activ_0, grad_0 = get_activation_and_grad_wrt_param(
            model, x, layer_of_interest_name, param_0
        )
        param_0 = param_0.clone()

        if optim == "sgd":
            lr_depth_scale = (L/base_depth)**(2*alpha - 1) if alpha != 0 else 1
            optimizer = torch.optim.SGD(
                model.parameters(),
                lr=lr * lr_depth_scale * N
            )
        elif optim == "adam":
            lr_depth_scale = (L/base_depth)**(alpha - 1) if alpha != 0 else 1
            eps_depth_scale = (L/base_depth)**(-alpha)
            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=lr * (1/np.sqrt(N)) * lr_depth_scale,
                eps=1e-8 * eps_depth_scale
            )
        elif optim == "scion":
            lr_depth_scale = (L/base_depth)**(alpha - 1) if alpha != 0 else 1
            if scion_method == "unconstrained_scion":
                optimizer = Scion(
                    optim_groups,
                    lr=lr * lr_depth_scale * (1/N),
                    unconstrained=True,
                    momentum=0.1
                )
            elif scion_method == "constrained_scion":
                optimizer = Scion(
                    optim_groups,
                    lr=lr * lr_depth_scale * (1/N),
                    constrained=True,
                    momentum=0.1
                )
            elif scion_method == "projected_scion":
                optimizer = Scion(
                    optim_groups,
                    lr=lr * lr_depth_scale * (1/N),
                    constrained=True,
                    projected=True,
                    momentum=0.1
                )

        criterion = nn.CrossEntropyLoss()

        # for name, param in model.named_parameters():
        #     param.requires_grad = (name == weight_of_interest_name)


        loss_history = []

        for step in range(n_steps):

            model.train()
            loss_step = 0.0

            # IMPORTANT: reset the iterator every epoch
            train_iter = iter(train_loader)

            for xt, yt in train_iter:
                xt, yt = xt.to(device), yt.to(device)

                optimizer.zero_grad()
                y_hat = model(xt)

                loss = criterion(y_hat, yt)
                loss.backward()
                optimizer.step()

                # print(f'loss: {loss.item()}')

                loss_step += loss.item()

            avg_loss = loss_step / len(train_loader)
            print(f'avg loss: {avg_loss}')
            loss_history.append(avg_loss)

        # Store the losses for this lr & method
        losses[(scion_method, lr)] = loss_history

    return losses

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False)
# losses_scion_alpha_1= find_best_lr(1.0, "unconstrained_scion", [2**i for i in range(-2, 5)], n_steps=10)

In [ ]:
train_iter = iter(train_loader)
x, y = next(train_iter)
x, y = x.to(device), y.to(device)
K = 2
D = 784
N_out = 10
N = 256
optim = 'scion'
lr = 0.1
max_steps = 1
Ls = [2, 4, 8, 16, 32, 64, 128, 256, 512, 
    # 1024, 
    # 2048, 
    # 4096
      ]
base_depth = Ls[0]
seeds = 50


weight_of_interest_name = 'branches.1.layers.0.weight'
layer_of_interest_name = 'branches.1'
activation_of_interest = 'branches.1.activs.0' # This is usd just for some testing

In [13]:
from tqdm import tqdm
import numpy as np


def experiment(alpha):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)


    delta_hs = np.zeros((seeds, len(Ls)))
    param_diff = np.zeros((seeds, len(Ls)))
    losses = np.zeros((len(Ls), max_steps))

    for s in tqdm(range(seeds)):
        for i,L in enumerate(Ls):
            train_iter = iter(train_loader) # re-set the loader
            
            model = ToyModel(D, N_out, N, L, K, alpha, base_depth)
            model.to(device)

            linear_weights = []
            first_weight = []
            last_weight = []
            radius_hidden_layer=1.0
            radius_first_layer=1.0
            radius_last_layer=1.0

            for name, module in model.named_modules():
                # print(name)
                if isinstance(module, nn.Linear):
                    if name == "readout":
                        last_weight.append(module.weight)
                    elif name == "readin":
                        first_weight.append(module.weight)
                    else:
                        linear_weights.append(module.weight)
                    # print(name, module.weight.dtype, torch.is_tensor(module.weight.data))

            # print("Nb of linear weight:", len(linear_weights))
            # print("Nb of last weight:", len(last_weight))
            # print("Nb of first weight:", len(first_weight))

            optim_groups = [{
                        'params': linear_weights,
                        'norm': 'Spectral',
                        'norm_kwargs': {'steps': 9}, # to stay consistent with the Muon baseline
                        # 'scale': [radius_hidden_layer]*len(linear_weights)
                        'scale': radius_hidden_layer
                    },
                    {   'params': first_weight,
                        'norm': 'Spectral',
                        'norm_kwargs': {},
                        # 'scale': [radius_first_layer]*len(first_weight)
                        'scale': radius_first_layer
                    },
                    {   'params': last_weight,
                        'norm': 'Sign',
                        'norm_kwargs': {},
                        # 'scale': [radius_last_layer]*len(last_weight)
                        'scale': radius_last_layer
                    }
            ]

            param_0 = get_param(model, weight_of_interest_name)
            activ_0, grad_0 = get_activation_and_grad_wrt_param(model, x, layer_of_interest_name, param_0)
            param_0 = param_0.clone()
        
            
            if optim == "sgd":
                lr_depth_scale = (L/base_depth)**(2*alpha - 1) if alpha != 0 else 1
                optimizer = torch.optim.SGD(model.parameters(), lr=lr * lr_depth_scale * N)
            elif optim == "adam":
                lr_depth_scale = (L/base_depth)**(alpha - 1) if alpha != 0 else 1
                eps_depth_scale = (L/base_depth)**(-alpha)
                optimizer = torch.optim.Adam(model.parameters(), lr=lr * 1/math.sqrt(N) * lr_depth_scale, eps=1e-8 * eps_depth_scale)
            elif optim == "scion":
                lr_depth_scale = (L/base_depth)**(alpha - 1) if alpha != 0 else 1
                optimizer = Scion(optim_groups, lr=lr * lr_depth_scale * 1/N , unconstrained=True, projected=False, momentum=0.1)
            criterion = nn.CrossEntropyLoss()
            
            #Only make the network trainable with respect to the weights of interest
            for name, param in model.named_parameters():
                if weight_of_interest_name != name:
                    param.requires_grad = False
                    
            step = 0
            for xt, yt in train_iter:
                xt, yt = xt.to(device), yt.to(device)
                optimizer.zero_grad()
                model.train()
                y_hat = model(xt)
                
                loss = criterion(y_hat, yt)

                loss.backward()
                optimizer.step()
                
                losses[i, step] = loss.item()
                
                step += 1
                if step >= max_steps:
                    break
            
            # make the network trainable with respect to all the weights (note: this does not affect the activations because we are not training from here anyway). 
            # This is done only to differentiate "activation_of_interest" with respect to the weights in the case K=1. 
            for name, param in model.named_parameters():
                param.requires_grad = True
            
            param_1 = get_param(model, weight_of_interest_name)
            activ_1, grad_1 = get_activation_and_grad_wrt_param(model, x, layer_of_interest_name, param_1)
        
            if K == 1:
                activ_11, _ = get_activation_and_grad_wrt_param(model, x, activation_of_interest, param_1, return_vector=True)
                assert torch.allclose(grad_1[0,:], math.sqrt(2/N) * activ_11[0,:], atol=1e-8), f"Gradient mismatch"
            
            ft = get_first_order_taylor((param_1-param_0), grad_0) 
            
            param_diff[s,i] = torch.norm(param_1 - param_0).detach().item()
            
            # activ_1 -> branch(t=1)
            # activ_0 -> branch(t=0) (zero-th order term)
            # ft -> (theta_1 - theta_0)^T \nabla branch |_{theta=\theta0} (first order term)
            
            ft_lin = activ_0 + ft
            delta_h = torch.abs((activ_1 - ft_lin)/ft_lin)
            delta_hs[s, i] = delta_h.detach().item()
        
    return delta_hs, param_diff, losses

In [14]:
delta_hs_1, param_diff_1 ,_ = experiment(alpha=1.0)
np.save(f'unconstrained_scion_lr_{lr}_delta_hs_1.npy', delta_hs_1)
np.save(f'unconstrained_scion_lr_{lr}_param_diff_1.npy', param_diff_1)

delta_hs_05, param_diff_05, _ = experiment(alpha=0.5)
np.save(f'unconstrained_scion_lr_{lr}_delta_hs_05.npy', delta_hs_05)
np.save(f'unconstrained_scion_lr_{lr}_param_diff_05.npy', param_diff_05)

delta_hs_0, param_diff_0, _ = experiment(alpha=0.0)
np.save(f'unconstrained_scion_lr_{lr}_delta_hs_0.npy', delta_hs_0)
np.save(f'unconstrained_scion_lr_{lr}_param_diff_0.npy', param_diff_0)

device: cuda


100%|██████████| 50/50 [07:02<00:00,  8.46s/it]


device: cuda


100%|██████████| 50/50 [07:00<00:00,  8.41s/it]


device: cuda


100%|██████████| 50/50 [07:09<00:00,  8.59s/it]
